# QUERY MANAGER AGENT

Setting up a multi-agent system to handle sales queries is a highly effective way to orchestrate complex data retrieval and reasoning tasks. Using Google’s Agent Development Kit (ADK), we can implement the **Agent-as-a-Tool** pattern, which allows a primary orchestration agent to delegate specialized tasks to sub-agents.

Since this will be running in a Jupyter Notebook, the architecture is broken down into modular components: defining your custom tools (Python functions), assembling your specialist agents, and finally creating the orchestrator that manages the workflow.

### Prerequisites for your Jupyter Notebook

First, you will need to install the necessary libraries in your notebook environment. Run this in your first cell:

```bash
!pip install google-adk pandas snowflake-connector-python O365

```

### Step 1: Define the Custom Tools

The ADK framework allows you to turn standard Python functions into tools simply by defining them with clear docstrings and type hints. The LLM reads these docstrings to understand what the tool does and when to invoke it.

#### 1.1: Configure your Gemini API Key

This notebook uses the [Gemini API](https://ai.google.dev/gemini-api/), which requires an API key.

In [12]:
%%capture
!pip install tabulate
!pip install O365

In [2]:
import os
from key import *

try:
    # Fetch the key from the local environment
    GOOGLE_API_KEY = dict_keys["GEMINI_API_KEY"]
    
    if not GOOGLE_API_KEY:
        raise ValueError("GEMINI_API_KEY not found. Please set it in your .env file or system environment variables.")

    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(f"🔑 Authentication Error: {e}")

✅ Gemini API key setup complete.


#### 1.4: Import ADK components

Now, import the specific components you'll need from the Agent Development Kit and the Generative AI library. This keeps your code organized and ensures we have access to the necessary building blocks.

In [3]:
from google.genai import types

from google.adk.agents import LlmAgent, Agent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search, AgentTool, ToolContext
from google.adk.code_executors import BuiltInCodeExecutor

print("✅ ADK components imported successfully.")

C:\Users\fabrb\anaconda3\envs\agent_env_v1\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


✅ ADK components imported successfully.


C:\Users\fabrb\anaconda3\envs\agent_env_v1\Lib\site-packages\google\adk\features\_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [4]:
import pandas as pd
import numpy as np
print("✅ Other libraries imported.")

✅ Other libraries imported.


### 1.4: Helper functions

Helper function that prints the generated Python code and results from the code execution tool:

In [5]:
def show_python_code_and_result(response):
    for i in range(len(response)):
        # Check if the response contains a valid function call result from the code executor
        if (
            (response[i].content.parts)
            and (response[i].content.parts[0])
            and (response[i].content.parts[0].function_response)
            and (response[i].content.parts[0].function_response.response)
        ):
            response_code = response[i].content.parts[0].function_response.response
            if "result" in response_code and response_code["result"] != "```":
                if "tool_code" in response_code["result"]:
                    print(
                        "Generated Python Code >> ",
                        response_code["result"].replace("tool_code", ""),
                    )
                else:
                    print("Generated Python Response >> ", response_code["result"])


print("✅ Helper functions defined.")

✅ Helper functions defined.


### 1.4 The Tool: Pulls the raw tabular data and returns it in a format the LLM can read (like a Markdown table or JSON).

The Analyst Agent (internal_data_specialist): Acts as the "second agent" you described. It receives that raw table, understands the schema, extracts the relevant financial metrics, and drops the noise.

The Writing Agent (sales_coordinator): Takes those extracted, clean data points and drafts the final comparison. You are correct that a dedicated "sentence-writing" agent in the middle is redundant; the coordinator can handle the synthesis directly.

Here is how the revised Python code looks reflecting this improved, realistic architecture:

1. The Realistic Data Tools
We will update the tools to mock returning tabular data (DataFrames converted to Markdown), exactly as they would when you use fetch_pandas_all() in the Snowflake connector.

In reality, a Snowflake query via the Python connector returns a cursor or a Pandas DataFrame—raw, tabular data with specific column headers, not a pre-formatted conversational sentence.

Your proposed architecture is a much more robust and realistic way to handle this in an agentic workflow. We don't necessarily need three separate agents for this specific pipeline; we can handle it beautifully with two layers by shifting the responsibilities.

Here is the refined architecture:

1. **The Tool:** Pulls the raw tabular data and returns it in a format the LLM can read (like a Markdown table or JSON).
2. **The Analyst Agent (`internal_data_specialist`):** Acts as the "second agent" you described. It receives that raw table, understands the schema, extracts the relevant financial metrics, and drops the noise.
3. **The Writing Agent (`sales_coordinator`):** Takes those extracted, clean data points and drafts the final comparison. You are correct that a dedicated "sentence-writing" agent in the middle is redundant; the coordinator can handle the synthesis directly.

Here is how the revised Python code looks reflecting this improved, realistic architecture:

### 1. The Realistic Data Tools

We will update the tools to mock returning tabular data (DataFrames converted to Markdown), exactly as they would when you use `fetch_pandas_all()` in the Snowflake connector.

In [6]:
def get_last_sales_email(sender_email: str) -> str:
    """Connects to Outlook and retrieves the body of the last email."""
    return "Mock Email: Please provide a performance update on Invesco QQQ and compare it to its main competitors."

def query_snowflake_fund_data(fund_name: str) -> str:
    """
    Executes a predefined query in Snowflake to collect raw performance data.
    Returns the tabular data as a Markdown string for the LLM to interpret.
    """
    # MOCKING REALITY: In production, this would be:
    # cursor.execute(query)
    # df = cursor.fetch_pandas_all()
    
    # We simulate a raw SQL table output
    raw_sql_data = {
        "FUND_TICKER": ["QQQ"],
        "FUND_NAME_LONG": ["Invesco QQQ Trust"],
        "YOY_RTN_PCT": [12.54],
        "AUM_USD_BN": [250.1],
        "INCEPTION_DT": ["1999-03-10"],
        "EXPENSE_RATIO_BPS": [20]
    }
    df = pd.DataFrame(raw_sql_data)
    
    # Returning as Markdown is the most reliable way for LLMs to read tabular data
    return df.to_markdown(index=False)

def get_invesco_characteristics(fund_name: str, file_path: str = "invesco_funds_mapping.xlsx") -> str:
    """
    Reads an Excel mapping file and returns the raw rows matching the fund.
    """
    # Mocking the Excel dataframe filtering
    excel_data = {
        "Fund": ["Invesco QQQ"],
        "Asset_Class": ["Large Cap Growth"],
        "Key_Feature": ["Tracks the Nasdaq-100 Index, excluding financial companies."]
    }
    df = pd.DataFrame(excel_data)
    
    return df.to_markdown(index=False)

### 2. The Updated Agent Logic

Now we adjust the `internal_data_specialist` to explicitly act as the data interpreter you suggested.

In [7]:
model = "gemini-3.1-flash-lite-preview"

In [8]:
# 1. The Internal Data Analyst (Your proposed "Second Agent")
internal_data_specialist = Agent(
    name="internal_data_specialist",
    model=model,
    description="Extracts and interprets raw tabular data from internal databases and files.",
    instruction="""You are a data analyst. Your job is to fetch and interpret raw tabular data.
    1. Read the sender's latest email to identify the fund.
    2. Use the Snowflake and Excel tools to pull the raw tables for that fund.
    3. Analyze the returned Markdown tables. Understand the schema (e.g., map YOY_RTN_PCT to Year-over-Year Performance).
    4. Extract ONLY the relevant features (Name, YoY performance, Top Holdings/Characteristics, AUM).
    5. Pass these clean, extracted data points to the orchestrator as a concise list. Do not write a long narrative.""",
    tools=[get_last_sales_email, query_snowflake_fund_data, get_invesco_characteristics]
)

# 2. The External Research Specialist (Remains the same)
research_agent = Agent(
    name="competitor_researcher",
    model=model,
    description="Conducts external web research on competing market funds.",
    instruction="""Research 3 main third-party competing funds for the given fund. 
    Focus exclusively on Europeant UCITS Funds and UCITS  ETFs in particular.
    Collect: Full Fund Name, YoY Performance, Top 10 Holdings. 
    Return a structured list of these data points.""",
    tools=[google_search]
)

# 3. The Chief Orchestrator (The Final "Writing" Agent)
sales_coordinator = Agent(
    name="sales_coordinator",
    model=model, 
    instruction="""You are the lead sales support coordinator. 
    1. Ask the internal_data_specialist to provide the extracted metrics for the fund requested in the email.
    2. Ask the competitor_researcher to provide metrics for third-party competitors.
    3. Take the raw data points from both specialists and write the final, professional response to the sales team.
    4. Ensure your final output is a well-formatted comparison outlining Invesco's competitive positioning.""",
    tools=[
        AgentTool(internal_data_specialist),
        AgentTool(research_agent)
    ]
)

## Step 2. Initialize the Runner with your top-level orchestrator agent

**Why InMemoryRunner?**

The Runner acts as the operational manager for the interaction. It is responsible for:

* Managing State: It keeps track of the conversation history (the Session) so the agent remembers previous steps and context. InMemoryRunner handles this directly in your computer's RAM, making it perfect for rapid prototyping and Jupyter Notebook testing.

* Event Orchestration: It handles the back-and-forth loop of the agent's reasoning. When the agent decides to pull Snowflake data, the Runner executes the tool, captures the Markdown table, and feeds it back to the LLM.

* Async Execution: It manages the asynchronous API calls to the Gemini models.

**How to implement it in your Jupyter Notebook**
Because ADK is heavily asynchronous, and modern Jupyter Notebooks natively support async/await at the top level of a cell, implementing the InMemoryRunner is very straightforward.

Here is how you wrap your sales_coordinator and launch the actual execution:

In [9]:
# 1. Initialize the Runner with your top-level orchestrator agent
runner = InMemoryRunner(agent=sales_coordinator)

sales_rep_email = "sales_rep@invesco.com"
prompt = f"Process the latest query from {sales_rep_email}"

print(f"Initiating workflow for query from: {sales_rep_email}...\n")

Initiating workflow for query from: sales_rep@invesco.com...



In [10]:
# 2. Execute the run 
# Jupyter allows you to use 'await' directly in the cell.
# Using run_debug is incredibly useful during prototyping as it outputs the agent's internal thought process.
events = await runner.run_debug(prompt)

# 3. The runner returns an event stream. 
# Depending on your specific ADK version, you typically extract the final text from the final event:
final_response = events[-1].content if events else "No response generated."
print("\n--- FINAL OUTPUT ---\n")
print(final_response)


 ### Created new session: debug_session_id

User > Process the latest query from sales_rep@invesco.com


sales_coordinator > Subject: Competitive Positioning Analysis: Invesco QQQ vs. Growth Market Peers

Dear Sales Team,

In response to the recent inquiry regarding the competitive standing of the **Invesco QQQ Trust (QQQ)** against market peers, please find the comparative data below. 

This analysis highlights QQQ’s institutional scale and performance, while noting the necessary regulatory distinctions for European-based prospects.

### Competitive Performance & Positioning

| Fund | Asset Class Focus | YoY Performance | Key Positioning |
| :--- | :--- | :--- | :--- |
| **Invesco QQQ** | Nasdaq-100 (Large Cap Growth) | **12.54%** | Market leader with $250.1B AUM; unmatched liquidity. |
| **UBS MSCI USA Growth UCITS** | MSCI USA Growth | ~11.8%* | Strong UCITS alternative for European growth exposure. |
| **iShares S&P 500 Info Tech UCITS** | Tech-focused Growth | ~14.2%* | High-conviction play; higher volatility due to sector concentration. |

*\*Estimated performance metrics for compar

Dear Sales Team,

In response to the recent inquiry regarding the competitive standing of the **Invesco QQQ Trust (QQQ)** against market peers, please find the comparative data below. 

This analysis highlights QQQ’s institutional scale and performance, while noting the necessary regulatory distinctions for European-based prospects.

### Competitive Performance & Positioning

| Fund | Asset Class Focus | YoY Performance | Key Positioning |
| :--- | :--- | :--- | :--- |
| **Invesco QQQ** | Nasdaq-100 (Large Cap Growth) | **12.54%** | Market leader with $250.1B AUM; unmatched liquidity. |
| **UBS MSCI USA Growth UCITS** | MSCI USA Growth | ~11.8%* | Strong UCITS alternative for European growth exposure. |
| **iShares S&P 500 Info Tech UCITS** | Tech-focused Growth | ~14.2%* | High-conviction play; higher volatility due to sector concentration. |

*\*Estimated performance metrics for comparative UCITS benchmarks.*

### Key Takeaways for Client Conversations

1.  **Dominant Scale:** With $250.1 Billion in AUM, the Invesco QQQ remains the primary vehicle for investors seeking direct exposure to the Nasdaq-100. Its liquidity profile is superior to most growth-factor ETFs, making it the preferred choice for institutional and active retail traders.
2.  **Regulatory Context (Important for EU clients):** Please note that U.S.-domiciled growth ETFs (like VUG or IUSG) are generally unavailable to European retail clients due to PRIIPs/KID requirements. When discussing growth strategies with these clients, please position the **Invesco EQQQ Nasdaq-100 UCITS ETF** as the compliant, direct counterpart to the QQQ strategy.
3.  **Core Growth Positioning:** While sector-specific funds (e.g., S&P 500 IT) may show higher short-term performance, QQQ provides a more diversified large-cap growth footprint by excluding financials while maintaining a broader tech/consumer/healthcare ecosystem compared to pure-play tech funds.

Please let me know if you require further data cuts or specific collateral to support your client meetings.

Best regards,


### Step 3: Real Mail connector:

Transitioning from a mock function to production code for Microsoft 365 (Outlook) requires connecting to the Microsoft Graph API. Since you already included the O365 library in your initial setup, we will use that as it provides a clean, Pythonic wrapper around the API.

Here is the actual, production-ready code to replace your mock function.

In [13]:
from O365 import Account
import logging

In [14]:
# Set up basic logging to catch errors during execution
logging.basicConfig(level=logging.ERROR)

def get_last_sales_email(sender_email: str) -> str:
    """
    Connects to an Outlook mailbox via Microsoft Graph API and retrieves 
    the text body of the most recent email from a specific sender.
    """
    # 1. Define your Azure AD Application Credentials
    # In production, NEVER hardcode these. Fetch them from environment variables or a secrets manager.
    CLIENT_ID = 'your_client_id_here'
    CLIENT_SECRET = 'your_client_secret_here'
    TENANT_ID = 'your_tenant_id_here' 
    
    credentials = (CLIENT_ID, CLIENT_SECRET)
    
    try:
        # 2. Initialize the Account
        # Using 'auth_flow='client_credential'' allows the agent to run in the background 
        # without requiring a web-browser login popup every time.
        account = Account(credentials, auth_flow='client_credential', tenant_id=TENANT_ID)
        
        # 3. Authenticate
        if not account.is_authenticated:
            account.authenticate()
            
        # 4. Access the specific mailbox
        # If your agent is reading a shared mailbox, specify it here. 
        # Otherwise, it defaults to the user associated with the app.
        mailbox = account.mailbox(resource='sales_inbox@yourcompany.com') 
        inbox = mailbox.inbox_folder()
        
        # 5. Build the query to filter for the specific sender
        query = inbox.new_query('from').equals(sender_email)
        
        # 6. Fetch the single most recent email matching the query
        # We order by received date descending and limit to 1
        messages = inbox.get_messages(limit=1, query=query, order_by='receivedDateTime desc')
        
        # Extract the first message from the generator
        for message in messages:
            # message.get_body_text() strips out the HTML and returns plain text for the LLM
            body_text = message.get_body_text()
            return f"Email Subject: {message.subject}\n\nEmail Body: {body_text}"
            
        return f"No recent emails found from {sender_email}."
        
    except Exception as e:
        logging.error(f"Failed to retrieve email: {str(e)}")
        # Returning the error to the LLM allows it to inform the user it couldn't fetch the data
        return f"System Error: Unable to access the mailbox to retrieve emails from {sender_email}. Please check system connections."